# Fine-tune Moirai-MoE-base on 20 years of multivariate S&P 500 data

This notebook:
1. Downloads 20 years of S&P 500 (`^GSPC`) daily OHLCV data (5 variates: Open, High, Low, Close, Volume)
2. Holds out the final ~1 trading year as a never-touched test set
3. Fully fine-tunes **Moirai-MoE-base** (935M params) on the remaining ~19 years, multivariate, using a GPU
4. Compares the fine-tuned model against the original zero-shot model on the held-out year
5. Produces plots + metrics

**Requires a GPU runtime**: Runtime -> Change runtime type -> T4 GPU (or better).

A 935M-parameter model fine-tuned with plain fp32 AdamW needs ~15GB just for optimizer
state -- right at a free T4's 16GB limit. This notebook uses `bitsandbytes`' 8-bit AdamW
(cuts optimizer memory ~4x) plus fp16 mixed precision to fit comfortably with headroom to spare.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> select a GPU, then re-run this cell.")

## 1. Clone the repo and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/Agrim-Nuware/MOIRAI-CODE.git"
if not os.path.isdir("repo"):
    !git clone $REPO_URL repo
%cd repo

In [ ]:
!pip install -q -e '.[notebook]'
!pip install -q bitsandbytes yfinance

# uni2ts pins numpy~=1.26, which downgrades Colab's preinstalled numpy 2.x.
# Colab's preinstalled pandas is built against numpy 2.x, so once numpy is
# downgraded the two are ABI-incompatible ("numpy.dtype size changed").
# Force-reinstall a matching pandas, then restart the runtime so every
# already-imported module (numpy got pulled in transitively by torch above)
# reloads consistently. This cell intentionally crashes/restarts the kernel --
# that's expected, not an error. After it restarts, just continue running
# from the next cell (installed packages and cloned files are unaffected).
!pip install -q --force-reinstall "numpy<2" "pandas>=2.0,<2.3"
import os

os.kill(os.getpid(), 9)

**The cell above deliberately restarts the Colab runtime** (to fix a numpy/pandas
version mismatch). You'll see a "session crashed" / "automatically restarted" notice --
that's expected. Once it restarts, just continue running the cells below in order;
you do **not** need to re-run the clone or pip install cells.

In [ ]:
%cd /content/repo
import numpy as np
import pandas as pd

print("numpy:", np.__version__, "| pandas:", pd.__version__)

In [ ]:
with open(".env", "w") as f:
    f.write("CUSTOM_DATA_PATH=dataset/uni2ts_storage\n")
print(open(".env").read())

## 2. Download 20 years of multivariate S&P 500 data and split train/val/test

In [ ]:
!python dataset/sp500/prepare_20y_data.py

In [ ]:
import json
import pandas as pd

with open("dataset/sp500/split_info.json") as f:
    split_info = json.load(f)

trainval_df = pd.read_csv("dataset/sp500/sp500_20y_trainval.csv", index_col=0, parse_dates=True)
train_length = split_info["train_length"]
date_offset = trainval_df.index[train_length - 1].strftime("%Y-%m-%d")

print("train_length:", train_length)
print("lightning val offset:", split_info["lightning_val_offset"])
print("lightning val length:", split_info["lightning_val_length"])
print("date_offset for CSV builder:", date_offset)

## 3. Build the uni2ts HF-format datasets (multivariate)

In [ ]:
!python -m uni2ts.data.builder.simple SP500 dataset/sp500/sp500_20y_trainval.csv \
  --dataset_type wide_multivariate --date_offset "{date_offset}" --freq B

## 4. Fine-tune Moirai-MoE-base (full fine-tune, GPU, fp16 + 8-bit AdamW)

Sized for a free-tier T4: ~26 training steps/epoch (834 windows / batch 32), early stopping
(patience=3 on validation loss) with a safety cap of 60 epochs (~1500 steps max). On a T4 this
is roughly **30-90 minutes**; expect it to often stop earlier once validation loss plateaus.

If you have a bigger GPU (A100 via Colab Pro+), you can afford a larger `train_dataloader.batch_size`
(e.g. 64-128) and/or a smaller `data.distance` (more windows per epoch) for a more thorough fine-tune
in a similar time budget.

In [ ]:
lightning_val_offset = split_info["lightning_val_offset"]
lightning_val_length = split_info["lightning_val_length"]

!python -m cli.train \
  -cp conf/finetune \
  exp_name=sp500_full_finetune \
  run_name=run1 \
  model=moirai_moe_1.0_R_base \
  model.patch_size=16 \
  model.context_length=512 \
  model.prediction_length=32 \
  model.num_training_steps=1500 \
  model.num_warmup_steps=50 \
  model.finetune_pattern=full \
  model.use_8bit_adam=true \
  model.lr=1e-5 \
  data=sp500 \
  data.patch_size=16 \
  data.context_length=512 \
  data.prediction_length=32 \
  data.mode=M \
  data.train_length={train_length} \
  data.distance=5 \
  val_data=sp500 \
  val_data.patch_size=16 \
  val_data.context_length=512 \
  val_data.prediction_length=32 \
  val_data.mode=M \
  val_data.offset={lightning_val_offset} \
  val_data.eval_length={lightning_val_length} \
  val_data.distance=32 \
  trainer.max_epochs=60 \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.precision=16-mixed \
  +trainer.log_every_n_steps=10 \
  train_dataloader.batch_size=32 \
  train_dataloader.num_workers=2 \
  val_dataloader.batch_size=16 \
  val_dataloader.num_workers=2

## 5. Evaluate: zero-shot vs fine-tuned, on the held-out final test year

In [ ]:
!python dataset/sp500/evaluate_finetuned.py \
  --context_length 512 --prediction_length 32 --num_samples 100

In [ ]:
from IPython.display import Image, display

print("Forecast comparison (Close price):")
display(Image("dataset/sp500/results_forecast_plot.png"))
print("\nMAPE comparison by variate:")
display(Image("dataset/sp500/results_metrics_bar.png"))
print("\nFine-tuning loss curve:")
display(Image("dataset/sp500/results_loss_curve.png"))

In [ ]:
import json

with open("dataset/sp500/results_metrics.json") as f:
    results = json.load(f)

print(f"{'variate':<8} {'zero-shot MAPE':>16} {'fine-tuned MAPE':>16}")
for v in ["Open", "High", "Low", "Close", "Volume"]:
    zs = results["zero_shot"][v]["mape"]
    ft = results["fine_tuned"][v]["mape"]
    print(f"{v:<8} {zs:>15.2f}% {ft:>15.2f}%")

## 6. (Optional) Save results back to your GitHub repo

Uncomment and fill in a [personal access token](https://github.com/settings/tokens) if you
want to push the fine-tuned checkpoint and plots back to your repo. Skip this if you'd rather
just download the files from the Colab file browser (left sidebar).

In [ ]:
# GITHUB_TOKEN = ""  # paste a token with repo write access, or leave blank to skip
# if GITHUB_TOKEN:
#     !git add dataset/sp500/results_*.png dataset/sp500/results_metrics.json
#     !git commit -m "Add Colab fine-tuning results"
#     !git push https://$GITHUB_TOKEN@github.com/Agrim-Nuware/MOIRAI-CODE.git HEAD:main